In [2]:
import pandas as pd
import numpy as np




data = pd.read_csv('Aditi_noondata_ (1).csv') 
data

C:\Users\akshaya\AppData\Local\Temp\ipykernel_213476\1121690478.py:7: DtypeWarning: Columns (87,99,150,152,153,167,170,181,182,183,186,187,188,189,194,197,206,219,341,353,354) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('Aditi_noondata_ (1).csv')


,id,pi_sog,pi_stw,true_wind_dir,true_wind_speed,relative_wind_speed,relative_wind_direction,caa,corrected_power,rw,...,rob_garbage_cat_a_plastic,rob_garbage_cat_b_organic,rob_garbage_cat_c_domestic,rob_garbage_cat_e_ashes,qty_sludge_disposed,steaming_time_m_e_lng,reefer_power_consumption,wind_direction_side,sea_direction_side,tot_sludge_removed
0,801,-100.00000,-100.00000,-68.00000,3.08664,2.64613,40.00000,0.48952,5800.00000,1736.69733,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21180,-8.95200,-9.50492,317.40157,6.45339,15.60975,-4.43245,0.54364,15997.49997,34602.89699,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,38134,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.61141,0.00000,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,64384,-22.18435,-20.06203,349.58788,2.96574,8.46249,-3.10361,0.56473,5133.06679,7865.70475,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,167751,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5453,1925153,-17.68807,-20.34339,-20.81459,3.66385,9.79017,8.98141,0.71577,6731.98239,17467.00690,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5454,1925704,-100.00000,-100.00000,0.00000,0.00000,0.00000,0.00000,0.61141,5284.00000,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5455,1925907,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.61141,0.00000,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5456,1926345,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.61141,0.00000,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:

#data.drop(21335, axis = 0, inplace = True)   # Unwanted time coming  486-90030  in steaming time column
#data.drop(data[data['sea_state'] == 90].index[0], inplace = True)  # Sea State 90 coming
data['sea_state'].value_counts()



sea_state
0.0    1418
3.0    1058
4.0     694
2.0     600
5.0     186
1.0      72
6.0      37
8.0       2
Name: count, dtype: int64

In [6]:
def find_greater_time(x, y):
    return x if x >= y else y

# Calculate sec_time
data['sec_time'] = [find_greater_time(i, j) for i, j in zip(
    data['steaming_time_me_rsdl_vls'].astype(float).fillna(0),
    data['steaming_time_me_rsdl_hs'].astype(float).fillna(0)
)]

# Calculate third_time
data['third_time'] = [find_greater_time(i, j) for i, j in zip(
    data['total_steaming_time'].astype(float).fillna(0),
    data['me_fuel_only_steaming_time'].astype(float).fillna(0)
)]

# Calculate me_actual_steaming_time
data['me_actual_steaming_time'] = [find_greater_time(i, j) for i, j in zip(
    data['third_time'].astype(float).fillna(0),
    data['sec_time'].astype(float).fillna(0)
)]

# Determine if the ME and BL are running
data['me_running'] = data['me_actual_steaming_time'].apply(lambda x: 1.0 if x > 0 else 0)
data['blr_running'] = data['bl_con'].apply(lambda x: 1.0 if x > 0 else 0)

# Count running auxiliary engines
def count_aux_running(row):
    count = 0
    for aux in ['steaming_time_aux_1', 'steaming_time_aux_2', 'steaming_time_aux_3', 'steaming_time_aux_4', 'steaming_time_aux_5']:
        if row[aux] > 0:
            count += 1
    return count

data['aux_running'] = data.apply(count_aux_running, axis=1)

# Define the columns of interest
mycols = [
    'imo', 'vessel', 'report_date_time', 'report_type', 'status', 'sea_state', 'cargo_total', 
    'cargo_total_teu', 'speed_by_log', 'speed_by_gps', 'rpm', 'slip', 'wind_speed', 'draft_aft', 'draft_fwd',
    'wave_height', 'sw_temp', 
    'steaming_time_me_rsdl_vls', 'steaming_time_me_rsdl_hs',  
    'steaming_time_me_dstlt_uls', 'steaming_time_me_mdo',
    'steaming_time_aux_1', 'steaming_time_aux_2', 'steaming_time_aux_3', 'steaming_time_aux_4', 
    'steaming_time_aux_5', 'ae_t_steaming', 'aux_running', 
    'blr_running', 'me_running',
    'me_actual_steaming_time',
    'total_steaming_time', 'me_fuel_only_steaming_time', 
    'miles_by_gps', 'manvrng_miles_by_gps',  
    'me_total_revs_counter',
    'draft', 'total_fo', 'me_con', 'ae_con', 'bl_con', 'total_co_2',
    'fuel_me_rsdl_hs', 'fuel_aux_rsdl_hs', 'fuel_boiler_rsdl_hs',
    'fuel_me_rsdl_vls', 'fuel_aux_rsdl_vls', 'fuel_boiler_rsdl_vls',
    'fuel_me_rsdl_uls', 'fuel_aux_rsdl_uls', 'fuel_boiler_rsdl_uls',
    'fuel_me_dstlt_vls', 'fuel_aux_dstlt_vls', 'fuel_boiler_dstlt_vls',
    'fuel_me_dstlt_uls', 'fuel_aux_dstlt_uls', 'fuel_boiler_dstlt_uls',
    'fuel_me_tnktnr_dstlt_vls', 'fuel_aux_tnktnr_dstlt_vls', 'fuel_boiler_tnktnr_dstlt_vls'
]

# Sort by vessel and report_date_time and select columns of interest
data = data.sort_values(['vessel', 'report_date_time'])[mycols]

# Extract date and time components
data['date_time'] = pd.to_datetime(data['report_date_time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
data['date'] = data['date_time'].dt.date.astype(str)
data['year'] = data['date_time'].dt.year
data['month_name'] = data['date_time'].dt.month_name()
data['quarter_no'] = data['date_time'].dt.quarter

# Display the first two rows of the dataframe
print(data.head(2))


          imo     vessel     report_date_time report_type    status  \
21    9235581  MSC ADITI  2017-07-09 23:54:00        SAIL   IN PORT   
1703  9235581  MSC ADITI  2017-07-10 01:00:00        COSP  DRIFTING   

      sea_state  cargo_total  cargo_total_teu  speed_by_log  speed_by_gps  \
21          0.0       4442.0            329.0           0.0           0.0   
1703        3.0       4442.0            329.0           0.0           0.0   

      ...  fuel_aux_dstlt_uls  fuel_boiler_dstlt_uls  \
21    ...                 NaN                    NaN   
1703  ...                 NaN                    NaN   

      fuel_me_tnktnr_dstlt_vls  fuel_aux_tnktnr_dstlt_vls  \
21                         NaN                        NaN   
1703                       NaN                        NaN   

      fuel_boiler_tnktnr_dstlt_vls           date_time        date  year  \
21                             NaN 2017-07-09 23:54:00  2017-07-09  2017   
1703                           NaN 2017-07-10 01:

In [8]:


# Function to adjust date based on "noon to noon" rule
def adjust_date(dt):
    if dt.hour < 12 or (dt.hour == 12 and dt.minute == 0 and dt.second == 0):
        return dt.date() - pd.Timedelta(days=1)
    else:
        return dt.date()


mydf = pd.DataFrame()
for i in data['vessel'].unique():
    df = data[data['vessel'] == i].sort_values([ 'vessel', 'report_date_time'])
    df['rev_count_diff'] = df['me_total_revs_counter'].diff() 
#     df['rev_count_rpm'] = df['me_total_revs_counter'].diff()/(24*60)

    df['sea_state'] = df['sea_state'].ffill().bfill()
    df['cargo_total'] = df['cargo_total'].ffill().bfill()
    df['cargo_total_teu'] = df['cargo_total_teu'].ffill().bfill()
    
    df['draft_aft'] = df['draft_aft'].replace(0, np.nan) 
    df['draft_aft'] = df['draft_aft'].ffill().bfill()
    
    df['draft_fwd'] = df['draft_fwd'].replace(0, np.nan)
    df['draft_fwd'] = df['draft_fwd'].ffill().bfill()
    
    
    df['adjusted_date'] = df['date_time'].apply(adjust_date)
    
    mydf = pd.concat([mydf, df], axis = 0)
    
 


data = mydf.reset_index(drop = True).copy()


#Total HS
data['total_hs'] =  data[['fuel_me_rsdl_hs' , 'fuel_aux_rsdl_hs' , 'fuel_boiler_rsdl_hs']].sum(axis=1)

#Total LS
data['total_ls'] =  data[["fuel_me_rsdl_vls",'fuel_aux_rsdl_vls',
                          'fuel_boiler_rsdl_vls','fuel_me_rsdl_uls',
                          'fuel_aux_rsdl_uls','fuel_boiler_rsdl_uls']].sum(axis = 1)

#Total ULS
data['total_uls'] =  data[['fuel_me_dstlt_vls' , 'fuel_aux_dstlt_vls' , 'fuel_boiler_dstlt_vls',
                           'fuel_me_dstlt_uls' ,  'fuel_aux_dstlt_uls' , 'fuel_boiler_dstlt_uls',
                           'fuel_me_tnktnr_dstlt_vls' , 'fuel_aux_tnktnr_dstlt_vls','fuel_boiler_tnktnr_dstlt_vls']].sum(axis=1)
  


data['derived_total_fo_mc'] = data[['me_con','ae_con','bl_con']].sum(axis=1)
data['derived_total_fo_type'] = data[['total_hs' , 'total_ls' , 'total_uls'  ]].sum(axis=1)
data['derived_total_fo_mc'].sum() ,data['derived_total_fo_type'].sum() , data['total_fo'].sum()



(47632.921, 46407.479, 69124.711)

In [10]:

def find_fuel(i, j):
    if (i >= j):
        return i
    else:
        return j
    
for i, j, k in zip(data['derived_total_fo_mc'].fillna(0), data['total_fo'].fillna(0), data.index):   
 
    data.loc[k, 'first_total_fo'] = find_fuel(i, j)

for i, j, k in zip(data['first_total_fo'].fillna(0), data['derived_total_fo_type'].fillna(0), data.index):   
 
    data.loc[k, 'actual_total_fo'] = find_fuel(i, j)


def find_distance(i, j):
    if i >= j:
        return i
    else:
        return j

for i, j, k in zip(data['miles_by_gps'].fillna(0), data['manvrng_miles_by_gps'].fillna(0), data.index):
 
    data.loc[k, 'distance'] = find_distance(i, j)



def find_speed(i, j):
    if (i == 0) & (j > 0):
        return j
    elif (j == 0) & (i > 0):
        return i
    else:
        
        return j

for i, j, k in zip(data['speed_by_log'].fillna(0), data['speed_by_gps'].fillna(0), data.index):   
 
    data.loc[k, 'speed'] = find_speed(i, j)

# Mean draft 0 replaced and ffill done above
data['mean_draft'] = (data['draft_aft'].fillna(0) + data['draft_fwd'].fillna(0))/2
data.head()

,imo,vessel,report_date_time,report_type,status,sea_state,cargo_total,cargo_total_teu,speed_by_log,speed_by_gps,...,total_hs,total_ls,total_uls,derived_total_fo_mc,derived_total_fo_type,first_total_fo,actual_total_fo,distance,speed,mean_draft
0,9235581,MSC ADITI,2017-07-09 23:54:00,SAIL,IN PORT,0.0,4442.0,329.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,2.2,2.2,0.00,0.00,6.7
1,9235581,MSC ADITI,2017-07-10 01:00:00,COSP,DRIFTING,3.0,4442.0,329.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,1.3,1.3,8.00,0.00,6.7
2,9235581,MSC ADITI,2017-07-10 12:00:00,NOON,AT SEA,3.0,4442.0,329.0,17.5,18.00,...,0.0,0.0,0.0,0.0,0.0,26.5,26.5,198.00,18.00,6.7
3,9235581,MSC ADITI,2017-07-10 23:00:00,EOSP,AT SEA,3.0,4442.0,329.0,17.1,17.27,...,0.0,0.0,0.0,0.0,0.0,25.8,25.8,189.97,17.27,6.7
4,9235581,MSC ADITI,2017-07-11 02:12:00,BRTH,DRIFTING,0.0,4442.0,329.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,4.0,4.0,24.00,0.00,6.7


In [11]:
data.shape

(5458, 77)

In [14]:
df1 = data[~data['status'].isna()].reset_index(drop = True)
df1.shape

(5317, 77)

In [16]:
df1['derived_speed'] = df1['distance']/df1['me_actual_steaming_time']

def find_speed(i, j):
    if (i == 0) & (j > 0):
        return j
    elif (j == 0) & (i > 0):
        return i
    else:
        
        return j

for i, j, k in zip(df1['derived_speed'].fillna(0), df1['speed'].fillna(0), df1.index):   
 
    df1.loc[k, 'actual_speed'] = find_speed(i, j)
    
    
df1['actual_speed'] = df1['actual_speed'].replace(np.inf, 0).round(2)


df1['derived_status'] = df1['status']
df1['derived_status'] = df1['derived_status'].replace("AT SEA", 'SEA-DRIFT')
df1['derived_status'] = df1['derived_status'].replace("DRIFTING", 'SEA-DRIFT')

In [18]:
df1['activity_time'] = (pd.to_timedelta(df1['date_time'].diff().map(lambda x :  '0 days 23:59:59'  if str(x) == '1 days 00:00:00' else str(x) ) ).dt.seconds/3600).round(1).fillna(0)
df1.to_excel("ADITI_CLASS_ALL_DATA.xlsx", index = False)

 

In [18]:
df1.shape

(5317, 81)